In [40]:
import numpy as np
import pandas as pd
from hmmlearn.hmm import GaussianHMM
from scipy.stats import norm
from scipy.linalg import expm
import yfinance as yf


#Black Scholes Option Pricing Formula 
def black_scholes_price(S0, K, T, r, sigma, option_type='call'):
    """
    Standard Black-Scholes option pricing formula.
    """
    d1 = (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    if option_type == ['call', 'Call']:
        price = S0 * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        price = K * np.exp(-r * T) * norm.cdf(-d2) - S0 * norm.cdf(-d1)
        
    return price

# Getting the data for Apple
ticker_symbol = "AAPL"
stock = yf.Ticker(ticker_symbol)
data = stock.history(period="5y")
df = data.copy()
df = df.reset_index()
df['Date'] = pd.to_datetime(df['Date'])
df = df.set_index('Date')

# Defining a Hidden Markov Model for Calculating the Volatility 

#--------------------------------------------------------------------
# We define two states of Volatility, Low Volatility (Bull Market) 
# and High Volatility (Bear Market), indicated by state 0 and 1
# respectively. 
#--------------------------------------------------------------------

def calculate_hmm_volatility(df, price_col='Close', T=0.5, n_regimes=2):
    prices = df[price_col].values
    log_returns = np.diff(np.log(prices)).reshape(-1, 1)
    
    # 1. Fit Gaussian HMM, Assuming Normal Distribution for each state
    hmm = GaussianHMM(n_components=n_regimes, covariance_type="diag", n_iter=1000, random_state=42)
    hmm.fit(log_returns)
    
    # Extract Daily Volatilities per regime
    daily_stds = np.sqrt(hmm.covars_.flatten())
    
    # Sort regimes by volatility: State 0 = Low Vol, State 1 = High Vol
    sort_idx = np.argsort(daily_stds)
    daily_stds = daily_stds[sort_idx]
    P_daily = hmm.transmat_[sort_idx][:, sort_idx]
    
    # Annualized volatilities per regime
    trading_days = 252
    sigma_regimes = daily_stds * np.sqrt(trading_days)
    
    # 2. Estimate initial state distribution (posterior probability of current day)
    hidden_states = hmm.predict(log_returns)
    state_map = {old_idx: new_idx for new_idx, old_idx in enumerate(sort_idx)}
    current_regime = state_map[hidden_states[-1]]
    
    # Initial state probability vector
    pi_0 = np.zeros(n_regimes)
    pi_0[current_regime] = 1.0
    
    # 3. Project average time spent in each regime over maturity T (n_steps)
    n_steps = int(T * trading_days)
    
    # Sum of transition probabilities over n_steps
    # Compute expected time proportion in each regime
    state_occupancy = np.zeros(n_regimes)
    current_p = np.eye(n_regimes)
    
    for _ in range(n_steps):
        state_occupancy += pi_0 @ current_p
        current_p = current_p @ P_daily
        
    regime_weights = state_occupancy / n_steps
    
    # 4. Compute expected forward variance & volatility
    # Var(r) = w_0 * sigma_0^2 + w_1 * sigma_1^2
    expected_variance = np.sum(regime_weights * (sigma_regimes**2))
    hmm_effective_vol = np.sqrt(expected_variance)
    
    print("=== HMM VOLATILITY ESTIMATION ===")
    print(f"Dataset End Date: {df.index[-1].strftime('%Y-%m-%d')}")
    print(f"Current Spot Price (S0): ${prices[-1]:.2f}")
    print(f"Active Regime: State {current_regime} ({'Low Vol' if current_regime == 0 else 'High Vol'})")
    print(f"Regime Annualized Volatilities: Low Vol = {sigma_regimes[0]:.2%}, High Vol = {sigma_regimes[1]:.2%}")
    print(f"Expected Time Allocation over T={T} years: State 0: {regime_weights[0]:.1%}, State 1: {regime_weights[1]:.1%}")
    print(f"HMM Effective Volatility (sigma_HMM): {hmm_effective_vol:.2%}\n")
    
    return prices[-1], hmm_effective_vol, sigma_regimes



if __name__ == "__main__":
    n_days = 756
    # Historical dataset ending 1 month ago (July 2026)
    dates = pd.date_range(end='2026-07-24', start = '2023-07-24', periods=n_days)
    # Parameters
    T = 0.5          # 6 Months to maturity
    r = 0.04         # 4% Risk-free rate
    strikes = [300, 305, 310, 315, 320]
    
    # 1. Calculate HMM-weighted Volatility
    S0, sigma_hmm, sigma_regimes = calculate_hmm_volatility(df, price_col='Close', T=T)
    
    # 2. Calculate Standard Historical Volatility (Simple 30-day trailing)
    daily_returns = np.diff(np.log(df['Close'].values))
    sigma_hist = np.std(daily_returns[-30:]) * np.sqrt(252)
    
    # 3. Price European Call Options across strikes
    results = []
    for K in strikes:
        price_hmm = black_scholes_price(S0, K, T, r, sigma=sigma_hmm, option_type='call')
        price_hist = black_scholes_price(S0, K, T, r, sigma=sigma_hist, option_type='call')
        
        results.append({
            'Strike ($)': K,
            'BS Price (HMM Vol)': round(price_hmm, 2),
            'BS Price (30d Hist Vol)': round(price_hist, 2),
            'Price Difference ($)': round(price_hmm - price_hist, 2)
        })
        
    results_df = pd.DataFrame(results)
    
    print("=== BLACK-SCHOLES OPTION PRICES COMPARISON ===")
    print(f"HMM Volatility Input:       {sigma_hmm:.2%}")
    print(f"30-Day Historical Vol Input: {sigma_hist:.2%}\n")
    print(results_df.to_string(index=False))

=== HMM VOLATILITY ESTIMATION ===
Dataset End Date: 2026-08-24
Current Spot Price (S0): $310.34
Active Regime: State 0 (Low Vol)
Regime Annualized Volatilities: Low Vol = 21.64%, High Vol = 54.62%
Expected Time Allocation over T=0.5 years: State 0: 86.4%, State 1: 13.6%
HMM Effective Volatility (sigma_HMM): 28.45%

=== BLACK-SCHOLES OPTION PRICES COMPARISON ===
HMM Volatility Input:       28.45%
30-Day Historical Vol Input: 32.29%

 Strike ($)  BS Price (HMM Vol)  BS Price (30d Hist Vol)  Price Difference ($)
        300               16.94                    20.08                 -3.15
        305               19.14                    22.37                 -3.23
        310               21.50                    24.79                 -3.29
        315               24.02                    27.35                 -3.33
        320               26.70                    30.05                 -3.35


In [1]:
import numpy as np
import scipy.stats as stats
from scipy.stats import norm

# =====================================================================
# 1. BAYESIAN HMM SAMPLER (GIBBS SAMPLING + FFBS)
# =====================================================================

class BayesianGaussianHMM:
    def __init__(self, n_regimes=2, alpha_prior=None, a0=2.0, b0=0.001):
        """
        Bayesian 2-State Gaussian HMM with conjugate priors.
        - Transition rows ~ Dirichlet(alpha)
        - Regime variances ~ Inverse-Gamma(a0, b0)
        """
        self.K = n_regimes
        
        # Dirichlet priors for transition matrix rows (flat uniform priors by default)
        if alpha_prior is None:
            self.alpha_prior = np.ones((self.K, self.K)) #Dirichlet(1,1) for all the P_i of the Transition Probability Matrix {i -> rows}
        else:
            self.alpha_prior = alpha_prior
            
        # Inverse-Gamma prior hyperparameters for regime variances
        self.a0 = a0
        self.b0 = b0

    def _forward_filter_backward_sample(self, returns, P, sigmas):
        """
        Step 1: Forward-Filtering Backward-Sampling (FFBS) to sample hidden states Z.
        """
        T = len(returns)
        K = self.K
        
        # 1. Forward Pass (Filtering)
        forward = np.zeros((T, K))
        
        # Initial state probabilities (uniform)
        init_prob = np.ones(K) / K
        
        # Calculate emission probabilities at t=0
        emission_0 = stats.norm.pdf(returns[0], loc=0, scale=sigmas)
        forward[0] = init_prob * emission_0
        forward[0] /= np.sum(forward[0])  # Normalize
        
        for t in range(1, T):
            emission_t = stats.norm.pdf(returns[t], loc=0, scale=sigmas)
            # Forward update: alpha_t = emission_t * (alpha_{t-1} @ P)
            forward[t] = emission_t * (forward[t-1] @ P)
            forward_sum = np.sum(forward[t])
            if forward_sum > 0:
                forward[t] /= forward_sum
            else:
                forward[t] = np.ones(K) / K

        # 2. Backward Pass (Sampling Z)
        states = np.zeros(T, dtype=int)
        
        # Sample final state T-1
        states[-1] = np.random.choice(K, p=forward[-1])
        
        # Backward sweep
        for t in range(T - 2, -1, -1):
            next_state = states[t + 1]
            # p(z_t | z_{t+1}, D) proportional to forward[t] * P[z_t, z_{t+1}]
            prob = forward[t] * P[:, next_state]
            prob /= np.sum(prob)
            states[t] = np.random.choice(K, p=prob)
            
        return states

    def fit_gibbs(self, returns, n_samples=2000, burn_in=500):
        """
        Runs Gibbs Sampling MCMC chain.
        """
        T = len(returns)
        K = self.K
        
        # Initialize parameters
        sigmas = np.array([0.01, 0.03])  # Initial std guesses (Low vol, High vol)
        P = np.array([[0.95, 0.05], [0.10, 0.90]])
        
        # Storage for posterior MCMC samples
        self.P_samples = []
        self.sigma_samples = []
        self.state_samples = []
        
        for iteration in range(n_samples):
            # ---------------------------------------------------------
            # Step A: Sample Hidden States (Z | P, sigmas, D)
            # ---------------------------------------------------------
            states = self._forward_filter_backward_sample(returns, P, sigmas)
            
            # ---------------------------------------------------------
            # Step B: Sample Transition Matrix P (Dirichlet Posterior)
            # ---------------------------------------------------------
            N_trans = np.zeros((K, K))
            for t in range(T - 1):
                N_trans[states[t], states[t+1]] += 1
                
            P_new = np.zeros((K, K))
            for i in range(K):
                dirichlet_param = self.alpha_prior[i] + N_trans[i]
                P_new[i] = np.random.dirichlet(dirichlet_param)
            P = P_new
            
            # ---------------------------------------------------------
            # Step C: Sample Variances sigma^2 (Inverse-Gamma Posterior)
            # ---------------------------------------------------------
            sigmas_new = np.zeros(K)
            for k in range(K):
                idx = (states == k)
                N_k = np.sum(idx)
                
                if N_k > 0:
                    sum_sq = np.sum(returns[idx] ** 2)
                    a_post = self.a0 + N_k / 2.0
                    b_post = self.b0 + sum_sq / 2.0
                    # Draw variance from Inverse-Gamma (via Inverse-Gamma scale draw)
                    var_k = stats.invgamma.rvs(a_post, scale=b_post)
                    sigmas_new[k] = np.sqrt(var_k)
                else:
                    sigmas_new[k] = sigmas[k]
                    
            # Ensure State 0 is always Low Vol, State 1 is High Vol (Enforce Identifiability)
            sort_order = np.argsort(sigmas_new)
            sigmas = sigmas_new[sort_order]
            P = P[sort_order][:, sort_order]
            states = np.array([0 if s == sort_order[0] else 1 for s in states])
            
            # Save post-burn-in draws
            if iteration >= burn_in:
                self.P_samples.append(P)
                self.sigma_samples.append(sigmas)
                self.state_samples.append(states)

        self.P_samples = np.array(self.P_samples)
        self.sigma_samples = np.array(self.sigma_samples)
        self.state_samples = np.array(self.state_samples)

# =====================================================================
# 2. BAYESIAN OPTION PRICING ENGINE
# =====================================================================

def black_scholes_call(S0, K, T, r, sigma):
    d1 = (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S0 * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

def compute_bayesian_option_price(model, S0, K_list, T=0.5, r=0.04):
    """
    Computes Posterior Predictive Option Prices by integrating Black-Scholes 
    over all MCMC samples.
    """
    n_draws = len(model.sigma_samples)
    trading_days = 252
    n_steps = max(int(T * trading_days), 1)
    
    option_price_draws = {K: [] for K in K_list}
    
    for i in range(n_draws):
        P_draw = model.P_samples[i]
        sigma_draw = model.sigma_samples[i] * np.sqrt(trading_days) # Annualize
        current_state = model.state_samples[i][-1]
        
        # Calculate expected time allocation for this MCMC draw
        pi_0 = np.zeros(2)
        pi_0[current_state] = 1.0
        
        state_occupancy = np.zeros(2)
        current_p = np.eye(2)
        for _ in range(n_steps):
            state_occupancy += pi_0 @ current_p
            current_p = current_p @ P_draw
            
        weights = state_occupancy / n_steps
        
        # Effective volatility for this draw
        eff_var = np.sum(weights * (sigma_draw**2))
        eff_vol = np.sqrt(eff_var)
        
        # Calculate BS option price for each strike
        for K in K_list:
            price = black_scholes_call(S0, K, T, r, eff_vol)
            option_price_draws[K].append(price)
            
    return option_price_draws

# =====================================================================
# 3. RUN SIMULATION & EVALUATION, Theortical Bayesian_HMM Option Pricing
# =====================================================================

if __name__ == "__main__":
    np.random.seed(42)
    
    # 1. Generate Synthetic Daily Log-Returns (2 Years)
    n_days = 504
    r_low = np.random.normal(0.0003, 0.008, 300)   # Low Vol (~12.7% annualized)
    r_high = np.random.normal(-0.0005, 0.022, 204) # High Vol (~34.9% annualized)
    returns = np.concatenate([r_low, r_high])
    
    # 2. Fit Bayesian Gaussian HMM via Gibbs Sampling
    print("Fitting Bayesian HMM via Gibbs Sampling...")
    model = BayesianGaussianHMM(n_regimes=2, a0=2.0, b0=0.001)
    model.fit_gibbs(returns, n_samples=1500, burn_in=500)
    
    # 3. Extract Posterior Summary Statistics
    avg_P = np.mean(model.P_samples, axis=0)
    avg_sigmas_daily = np.mean(model.sigma_samples, axis=0)
    avg_sigmas_ann = avg_sigmas_daily * np.sqrt(252)
    
    print("\n=== BAYESIAN HMM POSTERIOR ESTIMATES ===")
    print("Posterior Mean Transition Matrix P:")
    print(np.round(avg_P, 4))
    print(f"Posterior Mean Annualized Volatilities: State 0 = {avg_sigmas_ann[0]:.2%}, State 1 = {avg_sigmas_ann[1]:.2%}")
    
    # 4. Posterior Predictive Option Pricing across Strikes
    S0 = 100.0
    strikes = [90, 95, 100, 105, 110]
    T = 0.5  # 6 Months to Expiry
    
    price_draws = compute_bayesian_option_price(model, S0, strikes, T=T)
    
    print("\n=== POSTERIOR PREDICTIVE OPTION PRICES ===")
    print(f"{'Strike ($)':<10} {'Mean Price ($)':<15} {'95% Credible Interval ($)':<25}")
    print("-" * 55)
    
    for K in strikes:
        draws = price_draws[K]
        mean_p = np.mean(draws)
        lower_p = np.percentile(draws, 2.5)
        upper_p = np.percentile(draws, 97.5)
        print(f"{K:<10} ${mean_p:<14.2f} [${lower_p:.2f} - ${upper_p:.2f}]")

Fitting Bayesian HMM via Gibbs Sampling...

=== BAYESIAN HMM POSTERIOR ESTIMATES ===
Posterior Mean Transition Matrix P:
[[0.9928 0.0072]
 [0.0059 0.9941]]
Posterior Mean Annualized Volatilities: State 0 = 13.17%, State 1 = 34.63%

=== POSTERIOR PREDICTIVE OPTION PRICES ===
Strike ($) Mean Price ($)  95% Credible Interval ($)
-------------------------------------------------------
90         $15.50          [$14.04 - $16.61]
95         $12.40          [$10.69 - $13.67]
100        $9.77           [$7.91 - $11.13]
105        $7.57           [$5.68 - $8.96]
110        $5.79           [$3.96 - $7.15]


In [3]:
import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.stats import norm
import yfinance as yf

# =====================================================================
# 1. BAYESIAN GAUSSIAN HMM CLASS (GIBBS SAMPLING + FFBS)
# =====================================================================

class BayesianGaussianHMM:
    def __init__(self, n_regimes=2, alpha_prior=None, a0=2.0, b0=0.001):
        self.K = n_regimes
        self.alpha_prior = np.ones((self.K, self.K)) if alpha_prior is None else alpha_prior
        self.a0 = a0
        self.b0 = b0

    def _forward_filter_backward_sample(self, returns, P, sigmas):
        T = len(returns)
        K = self.K
        
        # 1. Forward Pass (Filtering)
        forward = np.zeros((T, K))
        init_prob = np.ones(K) / K
        emission_0 = stats.norm.pdf(returns[0], loc=0, scale=sigmas)
        
        forward[0] = init_prob * emission_0
        forward[0] /= (np.sum(forward[0]) + 1e-12)
        
        for t in range(1, T):
            emission_t = stats.norm.pdf(returns[t], loc=0, scale=sigmas)
            forward[t] = emission_t * (forward[t-1] @ P)
            f_sum = np.sum(forward[t])
            forward[t] = forward[t] / f_sum if f_sum > 0 else np.ones(K) / K

        # 2. Backward Pass (Sampling Z)
        states = np.zeros(T, dtype=int)
        states[-1] = np.random.choice(K, p=forward[-1])
        
        for t in range(T - 2, -1, -1):
            next_state = states[t + 1]
            prob = forward[t] * P[:, next_state]
            prob_sum = np.sum(prob)
            prob = prob / prob_sum if prob_sum > 0 else np.ones(K) / K
            states[t] = np.random.choice(K, p=prob)
            
        return states

    def fit_gibbs(self, returns, n_samples=2000, burn_in=500):
        T = len(returns)
        K = self.K
        
        # Initial parameters
        sigmas = np.array([0.008, 0.025])  # Initial std guesses
        P = np.array([[0.95, 0.05], [0.10, 0.90]])
        
        self.P_samples = []
        self.sigma_samples = []
        self.state_samples = []
        
        for iteration in range(n_samples):
            # Step A: Sample Hidden States
            states = self._forward_filter_backward_sample(returns, P, sigmas)
            
            # Step B: Sample Transition Matrix P
            N_trans = np.zeros((K, K))
            for t in range(T - 1):
                N_trans[states[t], states[t+1]] += 1
                
            P_new = np.zeros((K, K))
            for i in range(K):
                P_new[i] = np.random.dirichlet(self.alpha_prior[i] + N_trans[i])
            P = P_new
            
            # Step C: Sample Regime Variances
            sigmas_new = np.zeros(K)
            for k in range(K):
                idx = (states == k)
                N_k = np.sum(idx)
                if N_k > 0:
                    sum_sq = np.sum(returns[idx] ** 2)
                    a_post = self.a0 + N_k / 2.0
                    b_post = self.b0 + sum_sq / 2.0
                    var_k = stats.invgamma.rvs(a_post, scale=b_post)
                    sigmas_new[k] = np.sqrt(var_k)
                else:
                    sigmas_new[k] = sigmas[k]
                    
            # Enforce identifiability (State 0 = Low Vol, State 1 = High Vol)
            sort_order = np.argsort(sigmas_new)
            sigmas = sigmas_new[sort_order]
            P = P[sort_order][:, sort_order]
            states = np.array([0 if s == sort_order[0] else 1 for s in states])
            
            if iteration >= burn_in:
                self.P_samples.append(P)
                self.sigma_samples.append(sigmas)
                self.state_samples.append(states)

        self.P_samples = np.array(self.P_samples)
        self.sigma_samples = np.array(self.sigma_samples)
        self.state_samples = np.array(self.state_samples)

# =====================================================================
# 2. BLACK-SCHOLES PRICING ENGINE & POSTERIOR PREDICTIVE PRICER
# =====================================================================

def black_scholes_call(S0, K, T, r, sigma):
    d1 = (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T) + 1e-12)
    d2 = d1 - sigma * np.sqrt(T)
    return S0 * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

def price_bayesian_options(model, S0, strikes, T, r=0.045):
    n_draws = len(model.sigma_samples)
    price_draws = {K: [] for K in strikes}
    trading_days = 252
    n_steps = max(int(T * trading_days), 1)
    
    for i in range(n_draws):
        P_draw = model.P_samples[i]
        sigma_annual = model.sigma_samples[i] * np.sqrt(trading_days)
        
        # State probability distribution at t=0
        current_state = model.state_samples[i][-1]
        pi_t = np.zeros(model.K)
        pi_t[current_state] = 1.0
        
        # Calculate expected occupancy weights over horizon T
        state_occupancy = np.zeros(model.K)
        current_p = np.eye(model.K)
        for _ in range(n_steps):
            state_occupancy += pi_t @ current_p
            current_p = current_p @ P_draw
            
        weights = state_occupancy / n_steps
        
        # Convex mixture pricing across regimes to preserve fat-tail optionality
        for K in strikes:
            price_regime_0 = black_scholes_call(S0, K, T, r, sigma_annual[0])
            price_regime_1 = black_scholes_call(S0, K, T, r, sigma_annual[1])
            option_price = weights[0] * price_regime_0 + weights[1] * price_regime_1
            price_draws[K].append(option_price)
            
    return price_draws

# =====================================================================
# 3. YFINANCE DATA PIPELINE & MAIN EXECUTION
# =====================================================================

def main():
    ticker_symbol = "AAPL"
    print(f"Fetching 2-year market historical data for {ticker_symbol} from yfinance...")
    
    ticker = yf.Ticker(ticker_symbol)
    hist = ticker.history(period="2y")
    
    # Calculate daily log-returns
    returns = np.diff(np.log(hist['Close'].values))
    S0 = hist['Close'].iloc[-1]
    
    # Fetch option chains
    expirations = ticker.options
    target_exp = expirations[1]  # Pick near-term expiration (~1 month out)
    
    today = pd.Timestamp.now()
    exp_date = pd.Timestamp(target_exp)
    days_to_expiry = (exp_date - today).days
    T = days_to_expiry / 365.0
    
    opt_chain = ticker.option_chain(target_exp)
    calls = opt_chain.calls.copy()

    # Calculate midPrice first
    calls['midPrice'] = (calls['bid'] + calls['ask']) / 2.0

    # Relaxed filtering: keep options with volume/open interest or valid bid/ask quotes
    calls = calls[
        (calls['strike'] >= S0 * 0.90) & 
        (calls['strike'] <= S0 * 1.10) & 
        ((calls['bid'] > 0) | (calls['volume'] > 0) | (calls['openInterest'] > 0))
    ].copy()

    if calls.empty:
        print("Warning: No options met the strict filter criteria. Using all available calls in chain.")
        calls = opt_chain.calls.copy()
        calls['midPrice'] = (calls['bid'] + calls['ask']) / 2.0


    # Fit Bayesian HMM
    print("\nFitting Bayesian HMM via Gibbs Sampling (2000 iterations)...")
    model = BayesianGaussianHMM(n_regimes=2)
    model.fit_gibbs(returns, n_samples=2000, burn_in=500)
    
    # Print Posterior Summaries
    avg_P = np.mean(model.P_samples, axis=0)
    avg_sigmas = np.mean(model.sigma_samples, axis=0) * np.sqrt(252)
    
    print("\n" + "="*60)
    print(f"BAYESIAN POSTERIOR SUMMARY ({ticker_symbol})")
    print("="*60)
    print(f"Spot Price ($S_0$):            ${S0:.2f}")
    print(f"Option Expiration:           {target_exp} (T = {T:.3f} yrs / {days_to_expiry} days)")
    print(f"Regime 0 Annualized Vol:      {avg_sigmas[0]:.2%}")
    print(f"Regime 1 Annualized Vol:      {avg_sigmas[1]:.2%}")
    print("Posterior Transition Matrix P:")
    print(np.round(avg_P, 4))
    
    # Compute Bayesian Option Prices
    strikes = calls['strike'].values
    price_draws = price_bayesian_options(model, S0, strikes, T=T)
    
    results = []
    for _, row in calls.iterrows():
        K = row['strike']
        market_price = row['midPrice'] if row['midPrice'] > 0 else row['lastPrice']
        draws = price_draws[K]
        
        results.append({
            'Strike': K,
            'Market Price': market_price,
            'Bayesian Mean': np.mean(draws),
            '95% CI Lower': np.percentile(draws, 2.5),
            '95% CI Upper': np.percentile(draws, 97.5),
            'Market IV': row['impliedVolatility']
        })
        
    df_res = pd.DataFrame(results)
    
    print("\n" + "="*75)
    print("BAYESIAN MODEL VS. MARKET OPTIONS COMPARISON")
    print("="*75)
    print(f"{'Strike':<8} {'Market ($)':<12} {'Bayes Mean ($)':<16} {'95% Credible Interval ($)':<25} {'Market IV':<10}")
    print("-" * 75)
    for _, r in df_res.iterrows():
        ci_str = f"[${r['95% CI Lower']:.2f} - ${r['95% CI Upper']:.2f}]"
        print(f"{r['Strike']:<8.1f} ${r['Market Price']:<11.2f} ${r['Bayesian Mean']:<15.2f} {ci_str:<25} {r['Market IV']:.2%}")

if __name__ == "__main__":
    main()

Fetching 2-year market historical data for AAPL from yfinance...

Fitting Bayesian HMM via Gibbs Sampling (2000 iterations)...

BAYESIAN POSTERIOR SUMMARY (AAPL)
Spot Price ($S_0$):            $313.45
Option Expiration:           2026-08-31 (T = 0.008 yrs / 3 days)
Regime 0 Annualized Vol:      19.33%
Regime 1 Annualized Vol:      59.54%
Posterior Transition Matrix P:
[[0.9265 0.0735]
 [0.3994 0.6006]]

BAYESIAN MODEL VS. MARKET OPTIONS COMPARISON
Strike   Market ($)   Bayes Mean ($)   95% Credible Interval ($) Market IV 
---------------------------------------------------------------------------
282.5    $26.94       $31.07           [$31.06 - $31.13]         0.00%
285.0    $27.65       $28.57           [$28.56 - $28.67]         0.00%
287.5    $22.16       $26.08           [$26.06 - $26.24]         0.00%
290.0    $20.15       $23.59           [$23.56 - $23.85]         0.00%
292.5    $17.15       $21.11           [$21.07 - $21.49]         0.00%
295.0    $18.10       $18.63           [$